## Your pipeline, part 7: two more large models (training + inference only)
HW2, notebook 13 (Kaggle **GPU T4 x2**, ~2.3 h; a time guard keeps the lanes within 2.6 h).
The fusion of all members runs afterwards on CPU (notebook 14, no GPU quota).

**Why.** Notebook 11's best single model was YOLO11l (val 0.2531, better than notebook 10's whole 3-model fusion), and
notebook 12 showed that equal weights transfer to the leaderboard better than weights tuned on val (0.2964 vs 0.2938).
So: more strong members, all fused with equal weight.

| lane (GPU) | model | why | est. h (train + passes) |
|---|---|---|---|
| 0 | `L960s1`: YOLO11l, 960 px, seed 1 | same strength as 11's `L960`, different errors (default augmentation) | ~1.75 |
| 1 | `L1088`: YOLO11l, 1088 px | higher resolution for small objects (helped YOLO11m by +0.02) | ~2.2 |

Same training recipe as notebook 11 (box 12, dfl 2, 12 epochs, cosine schedule) and the same six-pass inference.
Estimates from notebook 11's measured times (`L960`: 81 min training + 22 min passes, run in parallel with another lane).

**Protections** (as in notebook 11): every job in its own process pinned to one GPU; memory check per model with automatic
batch halving; per-lane data folders; a model starts only if it fits within `BUDGET_H`; finished weights and passes are reused
from an attached earlier version of this notebook.

**Inputs:** competition; output of **01** (`vd_prep`). Accelerator **GPU T4 x2**, **Internet ON**.
**Outputs** for notebook 14: `passes_13_<model>_val.pkl.gz`, `passes_13_<model>_test.pkl.gz` (and `final_13_<model>.pt`).

In [1]:
%pip install -q ultralytics

In [2]:
import os

import os, io, json, time, math, random, shutil, contextlib
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

SEED = 0
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
import subprocess
try:
    _smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True).stdout.strip().splitlines()
except FileNotFoundError:
    _smi = []
GPUS = [(n.strip(), float(m) / 1024) for n, m in (l.split(",") for l in _smi)]      # (name, GB) per GPU
print("GPUs:", GPUS or "NONE -> choose the accelerator 'GPU T4 x2' in the session options")

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")

def find_all(pattern, depth=5):
    # search attached inputs at increasing nesting depth (Kaggle nests inputs differently per source)
    for d in range(depth):
        hit = sorted(INPUT.glob("/".join(["*"] * d + [pattern])))
        if hit:
            return hit
    return []

def find(pattern):
    hit = find_all(pattern)
    if not hit:
        raise FileNotFoundError(f"{pattern} not found under {INPUT}: check the attached inputs")
    return hit[0]

PREP = find("vd_prep/config.json").parent            # output of notebook 01
CFG = json.load(open(PREP / "config.json"))
TEST_CSV = find("test_images.csv"); ROOT = TEST_CSV.parent
VAL_DIR = find("val/annotations_coco.json").parent
FOLD = CFG["folds"]                                   # train clip -> fold id (GroupKFold by scene, from notebook 01)
METRICS = ["mAP", "AP50", "AP_s", "AP_m", "AP_l", "ms_per_img"]

import gc, gzip, pickle, sys, threading, ast
from multiprocessing import Pool
from ultralytics import YOLO, settings
settings.update({"wandb": False})

# ---- new members (lane = GPU); all get 04's best config, 12 epochs, cosine schedule (as notebook 10)
NEW = {"L960s1": dict(build="yolo11l.pt", imgsz=960, batch=4, seed=1),
       "L1088":  dict(build="yolo11l.pt", imgsz=1088, batch=4)}
EST_H = {"L960s1": 1.75, "L1088": 2.2}                     # train + passes hours (from notebook 11's measured times)
LANES_2GPU = {0: ["L960s1"], 1: ["L1088"]}
LANES_1GPU = {0: ["L960s1", "L1088"]}                      # one GPU: the time guard drops what does not fit
BUDGET_H = 2.6                                             # lanes must end within this many hours
EPOCHS = 12
DEV_ARG = 0                                                # device inside a worker (each worker sees only its GPU)
N_GPU = len(GPUS)
GPU_GB = min([g for _, g in GPUS] or [16.0])

# ---- inference and fusion (as in notebooks 07-10)
UP_SCALE, TILE_ZOOM, TILE_OVERLAP = 1.33, 1.5, 0.2
NMS_IOU, WBF_IOU, MAX_DET = 0.6, 0.55, 500
ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]
MERGE_WORKERS = max(1, (os.cpu_count() or 2))             # CPU processes for WBF / fusion (after the lanes)
VAL_STEP = 5

# ---- inputs: fail here, not after hours
need = {}                                                  # only the competition data and notebook 01's vd_prep
missing = [f"{f} (notebook {src})" for f, src in need.items() if not find_all(f)]
assert not missing, "attach the outputs that contain: " + ", ".join(missing)
done_new = [k for k in NEW if all(find_all(f"passes_13_{k}_{s}.pkl.gz") for s in ("val", "test"))]
assert N_GPU or len(done_new) == len(NEW), "no GPU: choose the accelerator 'GPU T4 x2' (or attach this notebook's finished output)"
if N_GPU == 1:
    print("WARNING: only one GPU -> one lane; the time guard will drop the model(s) that do not fit (choose 'GPU T4 x2')")

GPUs: [('Tesla T4', 15.0), ('Tesla T4', 15.0)]


### Data (same as notebooks 04-10)

In [3]:
def load_gt(path):
    # COCO ground truth; ignored regions (iscrowd=1) are copied to all 8 classes so they ignore detections of any class
    d = json.load(open(path))
    crowd = [a for a in d["annotations"] if a.get("iscrowd")]
    d["annotations"] = [a for a in d["annotations"] if not a.get("iscrowd")]
    nid = max([a["id"] for a in d["annotations"]], default=0) + 1
    for a in crowd:
        for c in range(1, 9):
            d["annotations"].append({**a, "id": nid, "category_id": c, "iscrowd": 1, "area": a["bbox"][2] * a["bbox"][3]})
            nid += 1
    gt = COCO()
    gt.dataset = d
    with contextlib.redirect_stdout(io.StringIO()):
        gt.createIndex()
    return gt

def coco_eval(gt, dets, img_ids):
    # standard COCO bbox metrics, up to 500 detections per image (as in the leaderboard)
    if not dets:
        return dict(mAP=0.0, AP50=0.0, AP_s=0.0, AP_m=0.0, AP_l=0.0)
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(gt, gt.loadRes(dets), "bbox")
        E.params.imgIds = list(img_ids)
        E.params.maxDets = [1, 10, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats                                  # s[1], s[3:6] already use maxDets[-1] = 500
    prec = E.eval["precision"][:, :, :, 0, -1]   # [IoU, recall, class] for all sizes at 500 dets (pycocotools' s[0] hardcodes 100)
    prec = prec[prec > -1]
    return dict(mAP=float(prec.mean()) if prec.size else -1.0, AP50=s[1], AP_s=s[3], AP_m=s[4], AP_l=s[5])

def write_submission(dets, path):
    # one row per test image: "class score x y w h ..." (0-based class, pixels, top-left corner, max 500 boxes)
    test = pd.read_csv(TEST_CSV)
    by = {}
    for d in dets:
        by.setdefault(d["image_id"], []).append(d)
    rows = []
    for iid in test.image_id:
        top = sorted(by.get(iid, []), key=lambda d: -d["score"])[:500]
        rows.append(" ".join(f"{d['category_id'] - 1} {d['score']:.4f} " + " ".join(f"{v:.1f}" for v in d["bbox"]) for d in top))
    sub = pd.DataFrame({"image_id": test.image_id, "PredictionString": rows})
    sub.to_csv(path, index=False)
    sample = find_all("sample_submission.csv")      # not shipped with this competition's data: ids come from test_images.csv
    if sample:
        assert list(sub.image_id) == list(pd.read_csv(sample[0]).image_id), "image ids differ from sample_submission.csv"
    print(f"saved {path}: {len(sub)} rows, {sum(map(bool, rows))} with detections, {sum(len(r.split()) // 6 for r in rows)} boxes")

In [4]:
# training frames exported by notebook 01 (ignored regions already grayed out)
TJ = json.load(open(PREP / "train_coco.json"))
IMGS = {i["id"]: i for i in TJ["images"]}
GT_TRAIN = load_gt(PREP / "train_coco.json")

def fold_ids(f):
    # (train ids, held-out ids) for fold f; f=None -> all frames for training
    return ([i for i, m in IMGS.items() if FOLD[m["video"]] != f],
            [i for i, m in IMGS.items() if FOLD[m["video"]] == f])

def copy_parallel(src, dst, workers=16):
    # copy a folder file by file with many threads (on network storage the per-file latency dominates)
    from concurrent.futures import ThreadPoolExecutor
    files = [p for p in Path(src).rglob("*") if p.is_file()]
    def cp(p):
        t = Path(dst) / p.relative_to(src)
        t.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(p, t)
    with ThreadPoolExecutor(workers) as ex:
        list(ex.map(cp, files))

# /kaggle/input is network storage (~5 MB/s, ~50 ms per file): copy the training images once to local disk
IMG_ROOT = PREP / "images" / "train"
if str(PREP).startswith("/kaggle/input"):
    local = Path("/tmp/vd_prep/images/train")
    if not (local.parent / ".copied").exists():
        t0 = time.time()
        copy_parallel(IMG_ROOT, local)
        (local.parent / ".copied").touch()
        print(f"copied {sum(1 for _ in local.iterdir())} training images to local disk in {time.time() - t0:.0f} s")
    IMG_ROOT = local

def train_path(i):
    return IMG_ROOT / IMGS[i]["file_name"]

# official val: raw frames at original resolution, every VAL_STEP-th frame of each clip (for speed)
VAL_GT = load_gt(VAL_DIR / "annotations_coco.json")
_idx = {(p.parent.name, p.name): p for p in (VAL_DIR / "images").rglob("*.jpg")}
_v = pd.DataFrame(VAL_GT.dataset["images"])
_v = _v[(_v.groupby("video").frame_id.rank(method="first").astype(int) - 1) % VAL_STEP == 0]
VAL_ITEMS = [(int(r.id), _idx[(r.video, Path(r.file_name).name)]) for r in _v.itertuples()]
VAL_IDS = [i for i, _ in VAL_ITEMS]
TEST_ITEMS = [(r.image_id, ROOT / r.file_name) for r in pd.read_csv(TEST_CSV).itertuples()]
print(f"{len(IMGS)} train frames | {len(VAL_ITEMS)} val frames | {len(TEST_ITEMS)} test frames")

copied 4868 training images to local disk in 24 s
4868 train frames | 571 val frames | 3811 test frames


In [5]:
import sys, subprocess

DEVICE_ID = globals().get("DEVICE_ID", 0)
# Ultralytics finds labels by swapping images -> labels in the path, and writes a .cache next to them
IMG_DIR = IMG_ROOT                                          # local copy of the training images (data cell)
LBL_DIR = IMG_DIR.parent.parent / "labels" / "train"
if not LBL_DIR.exists():
    copy_parallel(PREP / "labels" / "train", LBL_DIR)
JOB_DIR = Path("/tmp/yolo_jobs") if str(WORK).startswith("/kaggle") else WORK / "yolo_jobs"   # image lists, yamls, job files
JOB_DIR.mkdir(parents=True, exist_ok=True)

def write_data(name, tr_ids, va_ids):
    # Ultralytics data yaml with explicit image lists, so folds are just different lists
    for part, ids in (("train", tr_ids), ("val", va_ids)):
        (JOB_DIR / f"{name}_{part}.txt").write_text("\n".join(str(IMG_DIR / IMGS[i]["file_name"]) for i in ids))
    q = lambda x: json.dumps(str(x))                        # quoted path, safe for Windows backslashes
    p = JOB_DIR / f"{name}.yaml"
    p.write_text(f"path: {q(JOB_DIR)}\ntrain: {q(JOB_DIR / (name + '_train.txt'))}\nval: {q(JOB_DIR / (name + '_val.txt'))}\nnames: {json.dumps(CFG['classes'])}\n")
    return str(p)

# Every training and every prediction runs in its own short-lived Python process. When it exits, all of its
# GPU memory is released, so nothing piles up between runs (inside one process it did, and the GPU ran out).
WORKER = JOB_DIR / "ultra_worker.py"
WORKER.write_text("""
import json, sys
job = json.load(open(sys.argv[1]))
import torch
from ultralytics import YOLO, RTDETR, settings
settings.update({"wandb": False})
Cls = {"YOLO": YOLO, "RTDETR": RTDETR}[job["cls"]]
out = {}
if job["kind"] == "train":
    model = Cls(job["build"])
    if job.get("load"):
        model = model.load(job["load"])                     # built from a yaml: transfer the pretrained weights
    model.train(**job["kwargs"])
else:
    from ultralytics.utils import DEFAULT_CFG_DICT
    kw = dict(job["kwargs"])
    if job.get("fp16"):                                     # newer versions renamed half=True to quantize=16
        kw.update({"quantize": 16} if "quantize" in DEFAULT_CFG_DICT else {"half": True})
    model = Cls(job["weights"])
    items = json.load(open(job["items"]))
    dets, ms, CHUNK = [], [], 16
    # small chunks: Ultralytics 8.4 loads a whole list of paths as ONE batch (ignores `batch`) -> out of memory
    for k in range(0, len(items), CHUNK):
        chunk = items[k:k + CHUNK]
        for (iid, _), r in zip(chunk, model.predict([p for _, p in chunk], stream=True, **kw)):
            b = r.boxes
            xywh = b.xyxy.cpu().numpy()
            xywh[:, 2:] -= xywh[:, :2]
            for box, s, c in zip(xywh.tolist(), b.conf.tolist(), b.cls.tolist()):
                dets.append([iid, int(c) + 1, *[round(v, 2) for v in box], round(s, 5)])
            ms.append(r.speed["inference"] + r.speed["postprocess"])
    out["dets"], out["ms"] = dets, sum(ms) / max(len(ms), 1)
if torch.cuda.is_available():
    out["peak_gb"] = torch.cuda.max_memory_reserved() / 1e9
json.dump(out, open(job["out"], "w"))
""")

def run_worker(job, tag):
    # runs one job in a fresh process, prints its log here, returns its result
    job = {**job, "out": str(JOB_DIR / f"{tag}.out.json")}
    jp = JOB_DIR / f"{tag}.job.json"
    jp.write_text(json.dumps(job))
    proc = subprocess.Popen([sys.executable, str(WORKER), str(jp)], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"{tag}: the worker process failed, see its log above")
    return json.loads(Path(job["out"]).read_text())

def fit(spec, name, tr_ids, va_ids, **hp):
    kw = {"data": write_data(name, tr_ids, va_ids), "project": str(WORK / "runs"), "name": name, **{**COMMON, **hp}}
    res = run_worker({**spec, "kind": "train", "kwargs": kw}, name)
    (WORK / "curves").mkdir(exist_ok=True)
    shutil.copy(WORK / "runs" / name / "results.csv", WORK / "curves" / f"{TAG}_{name}.csv")     # per-epoch val curve
    # last epoch, not "best": best is picked on the held-out fold
    return WORK / "runs" / name / "weights" / "last.pt", res.get("peak_gb", float("nan"))

def detect(weights, items, imgsz):
    # returns COCO-style detections in original pixel coordinates and model ms per image (inference + postprocess)
    tag = f"predict_{Path(weights).parent.parent.name}_{len(items)}"
    ip = JOB_DIR / f"{tag}.items.json"
    ip.write_text(json.dumps([[i, str(p)] for i, p in items]))
    kw = dict(imgsz=imgsz, conf=0.001, iou=0.6, max_det=500, batch=4, verbose=False, device=DEVICE_ID)
    res = run_worker({"cls": MODEL_CLS.__name__, "kind": "predict", "weights": str(weights), "items": str(ip),
                      "kwargs": kw, "fp16": True}, tag)
    dets = [{"image_id": d[0], "category_id": d[1], "bbox": d[2:6], "score": d[6]} for d in res["dets"]]
    return dets, res["ms"]

def eval_fold(weights, ids, imgsz):
    dets, ms = detect(weights, [(i, IMG_DIR / IMGS[i]["file_name"]) for i in ids], imgsz)
    return {**coco_eval(GT_TRAIN, dets, ids), "ms_per_img": ms}

### The lanes: training + six-pass inference of each new model, one lane per GPU
Each job is a separate Python process that sees only its lane's GPU (`CUDA_VISIBLE_DEVICES`); its output goes to
`logs/<job>.log`. A status line is printed here every 10 minutes. This cell is the long one (~2.3 h).

In [6]:
# six-pass inference worker: the code of notebooks 07-10, run in its own process on one GPU
PASSES_WORKER = JOB_DIR / "passes_worker.py"
PASSES_WORKER.write_text("""
import json, sys, gzip, pickle, math, os
import numpy as np, cv2, torch
from torchvision.ops import batched_nms
from ultralytics import YOLO, settings
from ultralytics.utils import DEFAULT_CFG_DICT
settings.update({"wandb": False})
job = json.load(open(sys.argv[1]))
globals().update(job["consts"])
DEVICE_ID = job["device"]
PREC = ({"quantize": 16} if "quantize" in DEFAULT_CFG_DICT else {"half": True}) if torch.cuda.is_available() else {}
""" + """
def load_model(path):
    # model + the image size it was trained at
    ck = torch.load(path, map_location="cpu", weights_only=False)
    imgsz = int(np.max(ck.get("train_args", {}).get("imgsz", 960)))
    del ck
    return YOLO(str(path)), imgsz

def run(model, images, imgsz):
    # one model call on a few images (Ultralytics 8.4 puts a whole list into one batch); one (N, 6) array per image
    res = model.predict(images, imgsz=imgsz, conf=0.001, iou=0.6, max_det=MAX_DET, verbose=False, device=DEVICE_ID, **PREC)
    return [np.c_[r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()].astype(np.float32) for r in res]

def unflip(b, width):
    # boxes predicted on a horizontally flipped image -> original orientation
    b = b.copy(); b[:, [0, 2]] = width - b[:, [2, 0]]
    return b

def nms(b, iou):
    # class-wise NMS, best first, at most MAX_DET boxes (main process only)
    if not len(b):
        return b
    t = torch.from_numpy(b)
    keep = batched_nms(t[:, :4], t[:, 4], t[:, 5].long(), iou).numpy()
    return b[keep[:MAX_DET]]

def tile_grid(W, H):
    # overlapping crops of side max(W, H) / TILE_ZOOM (clipped to the frame), at least TILE_OVERLAP overlap
    T = int(max(W, H) / TILE_ZOOM)
    tw, th = min(T, W), min(T, H)
    def starts(size, t):
        n = max(1, math.ceil((size - t) / (t * (1 - TILE_OVERLAP))) + 1) if size > t else 1
        return np.linspace(0, size - t, n).round().astype(int)
    return [(x, y, x + tw, y + th) for y in starts(H, th) for x in starts(W, tw)]

ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]

def predict_views(model, imgsz, img):
    # all six passes of one frame, boxes in original pixel coordinates
    H, W = img.shape[:2]
    up = int(round(imgsz * UP_SCALE / 32) * 32)
    flip = np.ascontiguousarray(img[:, ::-1])
    out = {"full": run(model, [img], imgsz)[0], "full_flip": unflip(run(model, [flip], imgsz)[0], W),
           "up": run(model, [img], up)[0], "up_flip": unflip(run(model, [flip], up)[0], W)}
    grid = tile_grid(W, H)
    for name in ("tiles", "tiles_flip"):
        crops = [img[y0:y1, x0:x1] for x0, y0, x1, y1 in grid]
        if name == "tiles_flip":
            crops = [np.ascontiguousarray(c[:, ::-1]) for c in crops]
        parts = []
        for (x0, y0, x1, y1), b in zip(grid, run(model, crops, imgsz)):
            if name == "tiles_flip":
                b = unflip(b, x1 - x0)
            # drop boxes cut by an inner crop edge: the neighbouring crop (>= 20 % overlap) or the full pass has the whole object
            e = 2
            cut = (((b[:, 0] <= e) & (x0 > 0)) | ((b[:, 2] >= x1 - x0 - e) & (x1 < W)) |
                   ((b[:, 1] <= e) & (y0 > 0)) | ((b[:, 3] >= y1 - y0 - e) & (y1 < H)))
            b = b[~cut].copy(); b[:, [0, 2]] += x0; b[:, [1, 3]] += y0
            parts.append(b)
        out[name] = nms(np.concatenate(parts), NMS_IOU)       # remove duplicates from overlapping crops
    return out

def passes(model_path, tag, items):
    # prediction passes of one model on a set of frames; reused from this or an attached output, else computed and saved
    fname = f"passes_{tag}.pkl.gz"
    hit = [p for p in [WORK / fname] if p.exists()] + find_all(fname)
    if hit:
        with gzip.open(hit[0], "rb") as f:
            cache = pickle.load(f)
        if set(cache) == {i for i, _ in items}:
            print(f"reusing {hit[0]}")
            return cache
        print(f"{hit[0]} covers other frames, recomputing")
    model, imgsz = load_model(model_path)
    print(f"{tag}: {Path(model_path).name} trained at {imgsz} px, {len(items)} frames")
    cache = {iid: predict_views(model, imgsz, cv2.imread(str(p))) for iid, p in tqdm(items, desc=tag)}
    del model; gc.collect(); torch.cuda.empty_cache()
    with gzip.open(WORK / fname, "wb") as f:
        pickle.dump(cache, f, protocol=4)
    return cache

""" + """
model, imgsz = load_model(job["weights"])
for part in job["parts"]:
    items = json.load(open(part["items"]))
    cache = {}
    for k, (iid, p) in enumerate(items):
        cache[iid] = predict_views(model, imgsz, cv2.imread(p))
        if k % 500 == 0:
            print(part["out"], k, "/", len(items), flush=True)
    tmp = part["out"] + ".tmp"
    with gzip.open(tmp, "wb") as f:
        pickle.dump(cache, f, protocol=4)
    os.replace(tmp, part["out"])                            # complete file or none
out = {"peak_gb": torch.cuda.max_memory_reserved() / 1e9} if torch.cuda.is_available() else {}
json.dump(out, open(job["out"], "w"))
""")
LOG_DIR = WORK / "logs"; LOG_DIR.mkdir(exist_ok=True)
CONSTS = dict(MAX_DET=MAX_DET, UP_SCALE=UP_SCALE, TILE_ZOOM=TILE_ZOOM, TILE_OVERLAP=TILE_OVERLAP, NMS_IOU=NMS_IOU)
PRINT_LOCK = threading.Lock()
def say(*a):
    with PRINT_LOCK:
        print(time.strftime("%H:%M"), *a, flush=True)

def launch(script, job, tag, gpu):
    # one job in a fresh process that sees only GPU `gpu`; output -> logs/<tag>.log; returns the job's result
    job = {**job, "out": str(JOB_DIR / f"{tag}.out.json")}
    jp = JOB_DIR / f"{tag}.job.json"
    jp.write_text(json.dumps(job))
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)} if N_GPU else dict(os.environ)
    with open(LOG_DIR / f"{tag}.log", "w") as log:
        rc = subprocess.run([sys.executable, str(script), str(jp)], stdout=log, stderr=subprocess.STDOUT, env=env).returncode
    if rc != 0:
        tail = (LOG_DIR / f"{tag}.log").read_text(errors="replace").splitlines()[-40:]
        raise RuntimeError(f"{tag} failed (exit code {rc}); end of logs/{tag}.log:\n" + "\n".join(tail))
    return json.loads(Path(job["out"]).read_text())

def mine_or_attached(fname):
    return [p for p in [WORK / fname] if p.exists()] + find_all(fname)

def epochs_done(run):
    p = WORK / "runs" / run / "results.csv"
    return max(0, len(p.read_text().strip().splitlines()) - 1) if p.exists() else 0

TR_ALL, LOG_VAL = fold_ids(None)[0], fold_ids(0)[1][:200]  # all training frames; 200 frames only feed the training log
n_obj = {}
for a in TJ["annotations"]:
    if not a.get("iscrowd"):
        n_obj[a["image_id"]] = n_obj.get(a["image_id"], 0) + 1
DENSE = sorted(IMGS, key=lambda i: -n_obj.get(i, 0))[:64]   # most crowded frames, for the memory check
ITEM_FILES = {}
for split, items in (("val", VAL_ITEMS), ("test", TEST_ITEMS)):
    ITEM_FILES[split] = JOB_DIR / f"items_{split}.json"
    ITEM_FILES[split].write_text(json.dumps([[i, str(p)] for i, p in items]))

def write_lane_data(name, tr_ids, va_ids, gpu):
    # like write_data, but each lane reads the images through its own folder (symlinks), so Ultralytics writes its
    # label cache (labels/train.cache) per lane: two trainings starting together never read a half-written cache
    root = JOB_DIR / f"lane{gpu}"
    for kind, src in (("images", IMG_DIR), ("labels", LBL_DIR)):
        d = root / kind / "train"
        if not d.exists():
            d.parent.mkdir(parents=True, exist_ok=True)
            d.symlink_to(src, target_is_directory=True)
    img = root / "images" / "train"
    for part, ids in (("train", tr_ids), ("val", va_ids)):
        (JOB_DIR / f"{name}_{part}.txt").write_text("\n".join(str(img / IMGS[i]["file_name"]) for i in ids))
    q = lambda x: json.dumps(str(x))
    p = JOB_DIR / f"{name}.yaml"
    p.write_text(f"path: {q(JOB_DIR)}\ntrain: {q(JOB_DIR / (name + '_train.txt'))}\nval: {q(JOB_DIR / (name + '_val.txt'))}\nnames: {json.dumps(CFG['classes'])}\n")
    return str(p)

def train_kwargs(name, run, data, **over):
    c = NEW[name]
    workers = max(2, (os.cpu_count() or 4) // max(1, len(LANES)))
    kw = dict(data=data, project=str(WORK / "runs"), name=run, epochs=EPOCHS, batch=c["batch"], imgsz=c["imgsz"],
              workers=workers, device=DEV_ARG, seed=c.get("seed", SEED), warmup_epochs=2, close_mosaic=3,
              box=12.0, dfl=2.0, cos_lr=True, plots=False, exist_ok=True)
    if "scale" in c:
        kw["scale"] = c["scale"]
    return {**kw, **over}

def build_member(name, gpu):
    # final weights -> final_13_<name>.pt, then its val / test passes -> passes_13_<name>_{val,test}.pkl.gz
    final = WORK / f"final_13_{name}.pt"
    hit = mine_or_attached(final.name)
    run = f"final_13_{name}"
    if hit:
        if hit[0] != final:
            shutil.copy(hit[0], final)
        say(name, "final weights reused")
    elif epochs_done(run) >= EPOCHS:
        shutil.copy(WORK / "runs" / run / "weights" / "last.pt", final)
        say(name, "finished training run reused")
    else:
        bs = NEW[name]["batch"]
        while True:                                         # GPU memory check: one epoch on the 64 most crowded frames
            kw = train_kwargs(name, f"mem_{name}", write_lane_data(f"mem_{name}", DENSE, DENSE[:16], gpu),
                              epochs=1, warmup_epochs=0, close_mosaic=0, batch=bs)
            res = launch(WORKER, {"cls": "YOLO", "build": NEW[name]["build"], "kind": "train", "kwargs": kw}, f"mem_{name}_b{bs}", gpu)
            shutil.rmtree(WORK / "runs" / f"mem_{name}", ignore_errors=True)
            peak = res.get("peak_gb", 0.0)
            say(name, f"memory check: batch {bs} -> peak {peak:.1f} of {GPU_GB:.1f} GB")
            if peak < 0.85 * GPU_GB:
                break
            if bs <= 2:
                raise RuntimeError(f"{name}: too close to the GPU memory limit even at batch 2")
            bs //= 2
        NEW[name]["batch"] = bs
        t0 = time.time()
        say(name, f"training on {len(TR_ALL)} frames, batch {bs}")
        kw = train_kwargs(name, run, write_lane_data(run, TR_ALL, LOG_VAL, gpu), batch=bs)
        launch(WORKER, {"cls": "YOLO", "build": NEW[name]["build"], "kind": "train", "kwargs": kw}, run, gpu)
        shutil.copy(WORK / "runs" / run / "weights" / "last.pt", final)   # last epoch, as in 04-10
        (WORK / "curves").mkdir(exist_ok=True)
        shutil.copy(WORK / "runs" / run / "results.csv", WORK / "curves" / f"yolo11_{run}.csv")
        TIMES[name]["train_min"] = (time.time() - t0) / 60
        say(name, f"trained in {TIMES[name]['train_min']:.0f} min")
    parts = []
    for split in ("val", "test"):
        fname = f"passes_13_{name}_{split}.pkl.gz"
        hit = mine_or_attached(fname)
        if hit:
            if hit[0] != WORK / fname:
                shutil.copy(hit[0], WORK / fname)
        else:
            parts.append({"items": str(ITEM_FILES[split]), "out": str(WORK / fname)})
    if parts:
        t0 = time.time()
        launch(PASSES_WORKER, {"weights": str(final), "device": DEV_ARG, "consts": CONSTS, "parts": parts}, f"passes_{name}", gpu)
        TIMES[name]["passes_min"] = (time.time() - t0) / 60
        say(name, f"six passes on {', '.join(Path(p['out']).name for p in parts)} in {TIMES[name]['passes_min']:.0f} min")

def speed_ratio():
    # measured / estimated hours of the models built here so far (1 if none yet)
    r = [(t["train_min"] + t.get("passes_min", 0)) / 60 / EST_H[k] for k, t in TIMES.items() if "train_min" in t]
    return max(1.0, float(np.mean(r))) if r else 1.0

def lane(gpu, names):
    for name in names:
        ready = mine_or_attached(f"final_13_{name}.pt") and all(mine_or_attached(f"passes_13_{name}_{s}.pkl.gz") for s in ("val", "test"))
        el = (time.time() - T_START) / 3600
        need_h = 0 if ready else EST_H[name] * speed_ratio()
        if not ready and el + need_h > BUDGET_H:
            say(f"GPU {gpu}: {name} skipped by the time guard ({el:.1f} h elapsed + ~{need_h:.1f} h > {BUDGET_H} h)")
            STATE[name] = "skipped (time)"
            continue
        STATE[name] = "running"
        try:
            build_member(name, gpu)
            STATE[name] = "done"
        except Exception as e:                              # this lane goes on with its next model, the other lane is unaffected
            STATE[name] = "FAILED"
            say(f"GPU {gpu}: {name} FAILED:\n{e}")

LANES = LANES_2GPU if N_GPU >= 2 else LANES_1GPU
for b in sorted({c["build"] for c in NEW.values()}):        # download the pretrained weights once (lanes start at the same time)
    YOLO(b)
TIMES = {k: {} for k in NEW}
STATE = {k: "waiting" for k in NEW}
T_START = time.time()
threads = []
for gpu, names in LANES.items():
    t = threading.Thread(target=lane, args=(gpu, names), daemon=True)
    t.start(); threads.append(t)
    time.sleep(90 if N_GPU else 1)                          # stagger the starts (first-run downloads and caches)
last = 0
while any(t.is_alive() for t in threads):
    time.sleep(30)
    if time.time() - last > 600:
        last = time.time()
        say(f"{(time.time() - T_START) / 3600:.1f} h |", " | ".join(
            f"{k}: {STATE[k]}" + (f" epoch {epochs_done('final_13_' + k)}/{EPOCHS}" if STATE[k] == "running" else "") for k in NEW))
NEW_OK = [k for k in NEW if all((WORK / f"passes_13_{k}_{s}.pkl.gz").exists() for s in ("val", "test"))]
say(f"lanes finished in {(time.time() - T_START) / 3600:.2f} h | new members ready: {NEW_OK} | states: {STATE}")
if len(NEW_OK) < len(NEW):
    print("WARNING: not all new models finished; the fusion uses the ones that did (re-run with this output attached to add the rest)")
assert NEW_OK, "no new model finished: see the messages above and logs/"

21:24 L960s1 memory check: batch 4 -> peak 7.6 of 15.0 GB
21:24 L960s1 training on 4868 frames, batch 4
21:25 L1088 memory check: batch 4 -> peak 9.8 of 15.0 GB
21:25 L1088 training on 4868 frames, batch 4
21:27 0.1 h | L960s1: running epoch 0/12 | L1088: running epoch 0/12
21:37 0.2 h | L960s1: running epoch 1/12 | L1088: running epoch 1/12
21:47 0.4 h | L960s1: running epoch 3/12 | L1088: running epoch 2/12
21:57 0.6 h | L960s1: running epoch 4/12 | L1088: running epoch 3/12
22:07 0.7 h | L960s1: running epoch 5/12 | L1088: running epoch 4/12
22:17 0.9 h | L960s1: running epoch 7/12 | L1088: running epoch 5/12
22:27 1.1 h | L960s1: running epoch 8/12 | L1088: running epoch 6/12
22:37 1.2 h | L960s1: running epoch 10/12 | L1088: running epoch 7/12
22:47 1.4 h | L960s1: running epoch 11/12 | L1088: running epoch 8/12
22:51 L960s1 trained in 87 min
22:57 1.6 h | L960s1: running epoch 12/12 | L1088: running epoch 9/12
23:07 1.7 h | L960s1: running epoch 12/12 | L1088: running epoch 10/12

### Summary

In [7]:
summary = pd.DataFrame([{"model": k, **NEW[k], "state": STATE[k], **TIMES[k]} for k in NEW])
summary.to_csv(WORK / "results_13_summary.csv", index=False)
display(summary)
print("next: notebook 14 (CPU) with this output attached ->", ", ".join(f"passes_13_{k}_{{val,test}}.pkl.gz" for k in NEW_OK))

,model,build,imgsz,batch,seed,state,train_min,passes_min
0,L960s1,yolo11l.pt,960,4,1.0,done,87.149082,22.549359
1,L1088,yolo11l.pt,1088,4,NaN,done,110.981814,24.490677


next: notebook 14 (CPU) with this output attached -> passes_13_L960s1_{val,test}.pkl.gz, passes_13_L1088_{val,test}.pkl.gz


**Observations:**
- Both lanes finished in 2.29 h. YOLO11l at 960 px trained in 87 min (81 min in notebook 11), at 1088 px in 111 min, with 23-24 min of inference each.
- Peak GPU memory 7.6 GB (960 px) and 9.8 GB (1088 px) of 15 GB, so batch 4 fit for both.